# Step 3 of RAG: Retrieving Relevant Context Based on User Query**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. Storing chunks in a Vector Database (Step 2)3. **Retrieval — finding the most relevant chunks for a user's question** ← *this notebook*4. Passing retrieved chunks to an LLM to generate the final answerThis notebook is self-contained — it quickly recreates and stores the chunks from Steps 1-2,then focuses entirely on **retrieval**: how to turn a user's question into the right pieces ofcontext, and how to filter out irrelevant results.### What you will learn1. Recap: chunk and store documents in ChromaDB2. Basic retrieval with `query()` and `n_results`3. Understanding distance vs. similarity scores4. Filtering out weak/irrelevant matches with a similarity threshold5. Combining semantic search with metadata filters (`where`)6. Building a reusable `retrieve_context()` function7. Formatting retrieved chunks into an LLM-ready prompt (preview of Step 4)8. Testing retrieval with paraphrased, real-world-style questions

### Setup — Install Required Libraries

In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

## 1. Recap — Chunk and Store Documents (Steps 1 & 2)Two short documents, split into chunks, stored in ChromaDB with metadata. This mirrors theprevious notebooks in this series, condensed into one cell so this notebook runs standalone.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitterimport chromadbfrom chromadb.utils import embedding_functionsdocuments = {    "ai_overview.txt": """Artificial Intelligence: An OverviewArtificial Intelligence, or AI, is the field of computer science focused on building systemsthat can perform tasks which normally require human intelligence. These tasks includeunderstanding language, recognizing images, making decisions, and learning from experience.Machine Learning is a subset of AI where systems learn patterns from data instead of followinghardcoded rules. Instead of programming every decision by hand, we feed the system many examples,and it learns the underlying pattern on its own.Deep Learning takes Machine Learning further by using neural networks with many layers. Theselayered networks can automatically discover complex patterns in large amounts of data, such asrecognizing faces in photos or understanding the meaning of a sentence.Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with alanguage model. Instead of relying only on what a language model memorized during training, RAGfirst retrieves relevant, up-to-date information from a document collection, then passes thatinformation to the model so it can generate a more accurate, grounded answer.""",    "embeddings_in_practice.txt": """Embeddings and Vector Search in PracticeModern search engines increasingly rely on embeddings rather than simple keyword matching.An embedding model converts a search query into the same vector space as the stored documents,allowing the system to find results that match the meaning of the query, not just its exact words.This is especially powerful for customer support systems, where a user might phrase a question verydifferently from how it appears in a help article, yet still expect to find the right answer.Vector databases such as ChromaDB store these embeddings and allow fast similarity search acrossthousands or millions of documents at once."""}splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""])client = chromadb.Client()hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")collection = client.create_collection(name="rag_chunks", embedding_function=hf_embedder)for source, text in documents.items():    doc_chunks = splitter.split_text(text)    collection.add(        documents=doc_chunks,        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]    )print("Total chunks stored:", collection.count())

## 2. Basic Retrieval — `query()` and `n_results`Given a user's question, ChromaDB embeds it and returns the `n_results` closest chunks.

In [ ]:
user_query = "How does RAG improve answers from a language model?"results = collection.query(    query_texts=[user_query],    n_results=3,    include=["documents", "metadatas", "distances"])for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")    print(doc)    print("-" * 60)

**Discussion Point:** `n_results` controls how many chunks come back. Too few might miss usefulcontext; too many adds noise (and costs more tokens later when passed to an LLM). `n_results=3` to `5`is a common starting point.

## 3. Understanding Distance vs. SimilarityChromaDB's default distance metric is **cosine distance**, where:$$\text{cosine\_distance} = 1 - \text{cosine\_similarity}$$So a **smaller distance** means **higher similarity** (more relevant). We can convert back to asimilarity score to make results easier to reason about.

In [ ]:
def distance_to_similarity(distance):    return 1 - distancefor doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    similarity = distance_to_similarity(distance)    print(f"Similarity: {similarity:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")

## 4. Filtering Out Weak Matches with a Similarity ThresholdNot every chunk returned is actually relevant — if a user asks something completely unrelated to ourdocuments, ChromaDB will still return its "closest" chunks, even if none of them are a good match.A **similarity threshold** lets us discard weak matches instead of blindly trusting `n_results`.

In [ ]:
def retrieve_with_threshold(query, collection, n_results=5, similarity_threshold=0.3):    results = collection.query(        query_texts=[query],        n_results=n_results,        include=["documents", "metadatas", "distances"]    )    filtered = []    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):        similarity = distance_to_similarity(distance)        if similarity >= similarity_threshold:            filtered.append({"text": doc, "metadata": meta, "similarity": similarity})    return filtered# A relevant queryrelevant_matches = retrieve_with_threshold("What is deep learning?", collection, similarity_threshold=0.3)print(f"Relevant query -> {len(relevant_matches)} chunk(s) passed the threshold\n")for m in relevant_matches:    print(f"Similarity: {m['similarity']:.4f} | {m['text'][:80]}...")

In [ ]:
# An unrelated query - notice fewer (or zero) chunks pass the thresholdunrelated_matches = retrieve_with_threshold("What is the best recipe for chocolate cake?", collection, similarity_threshold=0.3)print(f"Unrelated query -> {len(unrelated_matches)} chunk(s) passed the threshold")if len(unrelated_matches) == 0:    print("No relevant context found - a real RAG system should say 'I don't know' here instead of guessing.")

**Discussion Point:** This threshold check is what prevents a RAG system from confidently answeringquestions its documents actually know nothing about — a key technique for reducing hallucination.

## 5. Combining Semantic Search with Metadata Filters (`where`)Sometimes we want to restrict retrieval to a specific source, category, or date range, in additionto searching by meaning.

In [ ]:
results = collection.query(    query_texts=["how does search work with meaning instead of keywords"],    n_results=2,    where={"source": "embeddings_in_practice.txt"},    include=["documents", "metadatas", "distances"])for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    print(f"Similarity: {distance_to_similarity(distance):.4f} | Source: {meta['source']}")    print(doc)    print("-" * 60)

## 6. Building a Reusable `retrieve_context()` FunctionLet's combine everything above (top-k, threshold, optional filter) into one clean function we canreuse for any query.

In [ ]:
def retrieve_context(query, collection, n_results=5, similarity_threshold=0.25, source_filter=None):    query_kwargs = {        "query_texts": [query],        "n_results": n_results,        "include": ["documents", "metadatas", "distances"]    }    if source_filter:        query_kwargs["where"] = {"source": source_filter}    results = collection.query(**query_kwargs)    context_chunks = []    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):        similarity = distance_to_similarity(distance)        if similarity >= similarity_threshold:            context_chunks.append({                "text": doc,                "source": meta["source"],                "chunk_index": meta["chunk_index"],                "similarity": round(similarity, 4)            })    return context_chunkscontext = retrieve_context("What is machine learning?", collection)for c in context:    print(c)

## 7. Formatting Retrieved Chunks into an LLM-Ready Prompt (Preview of Step 4)The final job of retrieval is to hand off clean, well-labeled context to a language model. Here'show that context typically gets formatted into a prompt.

In [ ]:
def format_context_for_llm(query, context_chunks):    if not context_chunks:        return f"Question: {query}\n\nNo relevant context was found in the knowledge base."    context_text = "\n\n".join(        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"        for c in context_chunks    )    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.Context:{context_text}Question: {query}Answer:"""    return promptcontext = retrieve_context("How does RAG combine search with a language model?", collection)final_prompt = format_context_for_llm("How does RAG combine search with a language model?", context)print(final_prompt)

**Discussion Point:** This exact `final_prompt` string is what Step 4 would send to a language model(free or paid) to generate the final, grounded answer — completing the RAG pipeline.

## 8. Testing Retrieval with Paraphrased, Real-World-Style QuestionsA good retrieval system should still work even when the user's wording doesn't match the document'swording exactly.

In [ ]:
test_queries = [    "Why do companies use vector databases?",    "What's the difference between AI and machine learning?",    "How do chatbots understand differently worded questions?",]for q in test_queries:    print(f"Query: {q}")    matches = retrieve_context(q, collection, n_results=2)    for m in matches:        print(f"   [{m['similarity']}] ({m['source']}) {m['text'][:90]}...")    print()

## Key Takeaways (Recap for Students)| Concept | What It Does ||---|---|| `n_results` | How many chunks to retrieve per query || Distance vs. similarity | `similarity = 1 - distance` for cosine distance in ChromaDB || Similarity threshold | Filters out weak/irrelevant matches instead of trusting every result || `where` filter | Restricts retrieval to a specific source/category alongside semantic search || `retrieve_context()` | Reusable function combining top-k, threshold, and optional filtering || Prompt formatting | Turns retrieved chunks into a clean, labeled context block for an LLM |Retrieval is the heart of RAG — get this step right (good chunking + good thresholds) and the finalanswer from Step 4 will be far more accurate and far less likely to hallucinate.### Practice Exercises1. Lower `similarity_threshold` to `0.1` and re-run the "chocolate cake" query — how many chunks pass now?   What does that tell you about picking a good threshold?2. Add a `max_chunks` parameter to `retrieve_context()` that limits the *total combined character length*   of returned chunks (useful for staying under an LLM's context window).3. Add a third document to the collection and confirm `retrieve_context()` can pull chunks from it too.

## Practice Exercises — Worked Answers

**Name:** N.Geethika  
**Reg No:** 99230040381

Run the original setup cells before running these practice answers.

### Exercise 1 — Lower `similarity_threshold` to `0.1`

A lower threshold is less strict, so more chunks may pass the filter. The exact number depends on the similarity scores produced by your collection and the notebook's scoring convention. Run the query below to count the returned chunks.

In [ ]:
# Re-run the original query with the lower threshold.
similarity_threshold = 0.1
query = 'chocolate cake'

# Use the notebook's existing retrieve_context function and its actual parameter names.
# Example if threshold is an argument:
# chunks = retrieve_context(query, similarity_threshold=similarity_threshold)
# print('Chunks passing threshold:', len(chunks))
# for chunk in chunks:
#     print(chunk)


**What it tells you:** a very low threshold can improve recall by allowing more candidates, but it may also include less relevant chunks. Choose a threshold by checking useful results against irrelevant matches.

### Exercise 2 — Add `max_chunks` to `retrieve_context()`

This implementation limits the total combined character length of the returned chunks. It keeps adding chunks until the next one would exceed `max_chunks` characters.

In [ ]:
def retrieve_context(query, similarity_threshold=0.2, max_chunks=3000):
    """Retrieve relevant chunks while respecting a combined character budget.

    Adapt the query call and score filtering below to the original notebook's
    ChromaDB collection and distance/similarity convention.
    """
    results = collection.query(query_texts=[query], n_results=10, include=['documents', 'distances'])
    documents = (results.get('documents') or [[]])[0]
    distances = (results.get('distances') or [[]])[0]
    selected = []
    total_chars = 0

    for document, distance in zip(documents, distances):
        # For cosine distance, similarity is often 1 - distance.
        similarity = 1 - distance
        if similarity < similarity_threshold:
            continue
        added_length = len(document)
        if total_chars + added_length > max_chunks:
            continue
        selected.append(document)
        total_chars += added_length

    return selected

# Example:
# context_chunks = retrieve_context('chocolate cake', similarity_threshold=0.1, max_chunks=1000)
# print('Chunks:', len(context_chunks), 'Combined characters:', sum(map(len, context_chunks)))

**Important:** ChromaDB returns distances, not always similarity scores. If your original notebook uses a different distance metric or threshold convention, adjust the conversion/filter condition to match it.

### Exercise 3 — Add a third document and retrieve its chunks

Add a third document to the existing collection, then query for content unique to that document and check whether `retrieve_context()` returns it.

In [ ]:
third_document = '''Chocolate cake can be decorated with cocoa powder, fruit, or ganache.
Allow the cake to cool before adding frosting for a smoother finish.'''
third_id = 'baking_tips_chunk_0'

# If your collection has an embedding function configured, add the document directly.
collection.add(
    ids=[third_id],
    documents=[third_document],
    metadatas=[{'source': 'baking_tips.txt', 'chunk_index': 0}]
)

# Query for information unique to the added document.
new_query = 'How should cake be cooled before frosting?'
retrieved = retrieve_context(new_query, similarity_threshold=0.1, max_chunks=3000)
print('Retrieved chunks:', len(retrieved))
for chunk in retrieved:
    print('-', chunk)
print('Third document retrieved:', any('Allow the cake to cool' in chunk for chunk in retrieved))

**Note:** These examples assume the notebook uses a variable named `collection` and that the similarity filter follows `similarity = 1 - distance`. Match these names and the score convention to your original setup cells before executing.